# Trabajo Práctico N° 1: Evaluación y Análisis de Redes Neuronales CBOW
**Asignatura**: Aprendizaje Automático Avanzado (UNAHUR)

Este notebook se destina a la visualización y análisis de resultados utilizando los modelos entrenados.

## Importación de librerías y módulos

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from collections import Counter
from codigo.modelo_cbow_cupy import cargar_modelo
from codigo.tokenizador import tokenizar_corpus

## Extracción de Vocabulario y Análisis de Distribución sobre el Corpus
Análisis de la estructura del vocabulario $|V|$  cálculo de las frecuencias de aparición de cada palabra sobre la secuencia completa del corpus de entrenamiento.

In [ ]:
# Cargar y tokenizar la secuencia COMPLETA del corpus de texto
ruta_corpus = "datos/corpus.txt"
tokens_completos = tokenizar_corpus(ruta_corpus=ruta_corpus, incluir_puntuacion_y_numeros=False)
print(f"Total de tokens en la secuencia del corpus completo: {len(tokens_completos):,}")

# Contar ocurrencias de cada palabra
frecuencias_corpus = Counter(tokens_completos)
palabras_unicas = list(frecuencias_corpus.keys())
tamanio_vocab = len(palabras_unicas)

print(f"Tamaño del vocabulario de palabras únicas (|V|): {tamanio_vocab:,}")

# Extraer métricas directas
frecuencias_palabras = list(frecuencias_corpus.values())
longitudes_palabras = [len(pal) for pal in palabras_unicas]

# Mostrar las 20 palabras más frecuentes
print("\nTop 20 palabras más frecuentes en el corpus completo:")
for pos, (palabra, freq) in enumerate(frecuencias_corpus.most_common(20), 1):
    print(f"  {pos:2d}. '{palabra}': {freq:,} apariciones")

# Mostrar las 20 palabras menos frecuentes
print("\nTop 20 palabras menos frecuentes en el corpus completo:")
for pos, (palabra, freq) in enumerate(frecuencias_corpus.most_common()[-20:], 1):
    print(f"  {pos:2d}. '{palabra}': {freq:,} apariciones")

# Filtrar el diccionario para quedarte solo con palabras de <= 'N' apariciones
cantidad_apariciones = 5000
palabras_hasta = {pal: freq for pal, freq in frecuencias_corpus.items() if freq <= cantidad_apariciones}

# Extraer las frecuencias para graficarlas si lo necesitás
frecuencias_hasta = list(palabras_hasta.values())

# Mostrar las 20 palabras más frecuentes dentro de este nuevo subgrupo
top_20_grupo = Counter(palabras_hasta).most_common(20)

print(f"\nTop 20 palabras en el rango de <= {cantidad_apariciones} repeticiones:")
for pos, (palabra, freq) in enumerate(top_20_grupo, 1):
    print(f"  {pos:2d}. '{palabra}': {freq:,} apariciones")

# Visualización Gráfica mediante Histogramas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de Frecuencia de Aparición
sns.histplot(frecuencias_palabras, bins=20, log_scale=True, color="#2b5c8f", ax=axes[0])
axes[0].set_title("Panorama General de Frecuencias", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Veces que se repite la palabra", fontsize=11)
axes[0].set_ylabel("Cantidad de Palabras Únicas", fontsize=11)
axes[0].xaxis.set_major_formatter(ticker.ScalarFormatter())

# Distribución por Longitud de Caracteres del Vocabulario
sns.histplot(longitudes_palabras, discrete=True, color="#d95f02", ax=axes[1])
axes[1].set_title("Distribución por Longitud de Caracteres en Vocabulario", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Longitud de la Palabra (caracteres)", fontsize=11)
axes[1].set_ylabel("Cantidad de Palabras Únicas", fontsize=11)

plt.tight_layout()
plt.show()

## Carga de un Modelo Entrenado (.npz)
Carga la estructura completa del modelo desde un archivo  y grafica la función de pérdida.

In [ ]:
ruta_modelo = "respaldos/modelo_cbow_m4_epoca_500_con_puntuacion.npz"
modelo = cargar_modelo(ruta_modelo)

In [ ]:
modelo['configuracion']

## Gráfico de Pérdida por Época

In [ ]:
# Graficar curva de pérdida por época
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))
epocas = range(1, len(modelo["historial_perdida"]) + 1)
plt.plot(epocas, modelo["historial_perdida"], color="#2b5c8f", linewidth=2.5, label='m=4')
plt.title("Evolución de la Pérdida de Entrenamiento por Época", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Comparación de curvas de pérdida entre modelos(.npz)
Carga modelos entrenados en formato .npz y grafica de forma superpuesta sus curvas de pérdida.

In [ ]:
# Cargar N modelos resguardados para comparativa y graficar
modelos_a_comparar = [
    {"ruta": "respaldos/modelo_cbow_m4_epoca_500_con_puntuacion.npz", "label": "m=4 con", "color": "#2b5c8f", "linestyle": "-"},
    {"ruta": "respaldos/modelo_cbow_m5_epoca_500_con_puntuacion.npz", "label": "m=5 con", "color": "#d95f02", "linestyle": "--"},
    {"ruta": "respaldos/modelo_cbow_m4_epoca_500_sin_puntuacion.npz", "label": "m=4 sin", "color": "#1b9e77", "linestyle": "-."},
    {"ruta": "respaldos/modelo_cbow_m5_epoca_500_sin_puntuacion.npz", "label": "m=5 sin", "color": "#7570b3", "linestyle": ":"},
    {"ruta": "respaldos/modelo_cbow_m5_epoca_1000_sin_puntuacion.npz", "label": "m=5 sin", "color": "#a570b3", "linestyle": "-"}
]

plt.figure(figsize=(10, 5))

for config in modelos_a_comparar:
    ruta = config["ruta"]
    label = config["label"]
    color = config.get("color")
    linestyle = config.get("linestyle")

    try:
        modelo = cargar_modelo(ruta)
        epocas = range(1, len(modelo["historial_perdida"]) + 1)
        plt.plot(epocas, modelo["historial_perdida"], label=label, color=color, linestyle=linestyle, linewidth=2)
    except FileNotFoundError:
        print(f"Advertencia: El modelo en la ruta '{ruta}' no fue encontrado. Saltando este modelo.")
    except Exception as e:
        print(f"Error al cargar o procesar el modelo '{ruta}': {e}. Saltando este modelo.")

plt.title("Comparativa de Curvas de Pérdida", fontsize=14, fontweight="bold")
plt.xlabel("Época", fontsize=12)
plt.ylabel("Pérdida Promedio", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Búsqueda de Similaridad de Palabras
Evalúa palabras similares calculando el producto interno o la similaridad de coseno.

In [ ]:
# @title Funcion de búsqueda de similaridad
def buscar_palabras_similares(
    palabra_buscada: str,
    W: np.ndarray,
    vocabulario_palabras: np.ndarray,
    top_k: int = 10,
    tipo_similaridad: str = "producto_interno"
) -> list[tuple[str, float]]:
    """
    Busca las palabras más similares a una palabra objetivo dada basándose en las filas de la matriz W.

    :param palabra_buscada: Palabra objetivo ingresada en texto.
    :param W: Matriz de pesos de entrada (|V| x N).
    :param vocabulario_palabras: Arreglo de cadenas de texto con las |V| palabras del vocabulario.
    :param top_k: Cantidad de vecinos más cercanos a retornar.
    :param tipo_similaridad: 'producto_interno' o 'coseno'.
    :return: Lista de tuplas (palabra, puntaje).
    """
    palabra_normalizada = palabra_buscada.lower().strip()
    vocab_lista = list(vocabulario_palabras)

    if palabra_normalizada not in vocab_lista:
        print(f"Advertencia: La palabra '{palabra_buscada}' no se encuentra en el vocabulario.")
        palabra_normalizada = "<UNK>"

    indice_objetivo = vocab_lista.index(palabra_normalizada)

    if hasattr(W, "get"):
        matriz_W = W.get()
    else:
        matriz_W = np.asarray(W)

    vector_palabra = matriz_W[indice_objetivo, :]

    if tipo_similaridad == "producto_interno":
        puntajes = np.dot(matriz_W, vector_palabra)
    elif tipo_similaridad == "coseno":
        norma_vector = np.linalg.norm(vector_palabra)
        normas_matriz = np.linalg.norm(matriz_W, axis=1)
        producto_punto = np.dot(matriz_W, vector_palabra)
        puntajes = producto_punto / (normas_matriz * norma_vector + 1e-12)
    else:
        raise ValueError(f"Tipo de similaridad no soportado: '{tipo_similaridad}'")

    indices_ordenados = np.argsort(puntajes)[::-1]

    resultados = []
    for indice_candidato in indices_ordenados:
        if indice_candidato == indice_objetivo:
            continue
        resultados.append((vocabulario_palabras[indice_candidato], float(puntajes[indice_candidato])))
        if len(resultados) >= top_k:
            break

    return resultados

In [ ]:
ruta_modelo = "respaldos/modelo_cbow_m4_epoca_500_sin_puntuacion.npz"
modelo = cargar_modelo(ruta_modelo)

# Probar la búsqueda de similaridad sobre el modelo cargado (Producto Interno y Similitud Coseno)
palabra_test = "él"

# Obtener la cantidad de repeticiones de la palabra buscada
frecuencia_palabra_test = frecuencias_corpus.get(palabra_test.lower(), 0)
print(f"La palabra '{palabra_test}' aparece {frecuencia_palabra_test:,} veces en el corpus.\n")

similares_pi = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="producto_interno"
)

similares_cos = buscar_palabras_similares(
    palabra_buscada=palabra_test,
    W=modelo["W"],
    vocabulario_palabras=modelo["vocabulario_palabras"],
    top_k=10,
    tipo_similaridad="coseno"
)

print(f"=== Palabras más similares a '{palabra_test}' (Producto Interno) ===")
for pal, score in similares_pi:
    print(f" - {pal:15s}: {score:.4f}")

print(f"\n=== Palabras más similares a '{palabra_test}' (Similitud Coseno) ===")
for pal, score in similares_cos:
    print(f" - {pal:15s}: {score:.4f}")


## Evaluación de analogía
Evalúa analogías entre palabras (A - B + C = D).

In [ ]:
# @title Funcion de evaluación de analogía
def evaluar_analogia(palabra_a, palabra_b, palabra_c, W, vocab_lista, top_k=5):
    """
    Evalúa A - B + C = D. Ejemplo: rey - hombre + mujer = reina.
    """
    a, b, c = palabra_a.lower(), palabra_b.lower(), palabra_c.lower()
    vocab_list = list(vocab_lista)

    for p in [a, b, c]:
        if p not in vocab_list:
            return f"Error: '{p}' no está en el vocabulario."

    idx_a, idx_b, idx_c = vocab_list.index(a), vocab_list.index(b), vocab_list.index(c)

    matriz_W = W.get() if hasattr(W, "get") else np.asarray(W)

    # Álgebra de vectores
    vec_res = matriz_W[idx_a] - matriz_W[idx_b] + matriz_W[idx_c]

    # Similitud coseno
    normas_matriz = np.linalg.norm(matriz_W, axis=1)
    norma_res = np.linalg.norm(vec_res)
    puntajes = np.dot(matriz_W, vec_res) / (normas_matriz * norma_res + 1e-12)

    indices_ordenados = np.argsort(puntajes)[::-1]

    resultados = []
    for idx in indices_ordenados:
        if idx not in [idx_a, idx_b, idx_c]: # Ignorar inputs
            resultados.append((vocab_list[idx], float(puntajes[idx])))
            if len(resultados) >= top_k:
                break

    return resultados

In [ ]:
evaluar_analogia("padre", "hombre", "mujer", modelo["W"], modelo["vocabulario_palabras"])